# 🛡️ SecureOps Assistant — Basic RAG Pipeline

A minimal, end-to-end RAG pipeline for industrial (OT/ICS) cybersecurity Q&A.

**Steps:**
0. Install dependencies & connect to Gemini
1. Download the corpus (NIST PDFs + CISA ICS advisories)
2. Parse & chunk the documents
3. Embed & index in ChromaDB
4. Retrieve relevant chunks
5. Generate a grounded, cited answer (refuses when unsupported)

## Step 0 — Install dependencies & connect to Gemini

In [ ]:
# Install the libraries we need (quiet).
%pip install -q chromadb sentence-transformers pypdf google-genai requests beautifulsoup4 lxml
print("✅ Dependencies installed")

In [ ]:
import os

# Load the Gemini API key: from Colab Secrets if available, else from env var, else prompt.
API_KEY = os.environ.get("GOOGLE_API_KEY")
try:
    from google.colab import userdata  # only exists inside Colab
    API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    pass
if not API_KEY:
    from getpass import getpass
    API_KEY = getpass("Paste your Gemini API key (input hidden): ")

# Create the Gemini client and pick a free-tier model.
from google import genai
client = genai.Client(api_key=API_KEY)
GEN_MODEL = "gemini-2.5-flash"

# Smoke test: one tiny call to confirm the key works.
r = client.models.generate_content(model=GEN_MODEL, contents="Reply with exactly: OK")
print("✅ Gemini connected:", r.text.strip())

## Step 1 — Download the corpus

Two public-domain NIST PDFs plus a sample of CISA ICS advisories (with a bundled fallback so the notebook always runs).

In [ ]:
import requests, pathlib

# Folder to hold the downloaded corpus.
CORPUS_DIR = pathlib.Path("corpus")
CORPUS_DIR.mkdir(exist_ok=True)
HEADERS = {"User-Agent": "Mozilla/5.0 (SecureOps student notebook)"}

# The two NIST PDFs.
PDFS = {
    "nist_sp800_82r3.pdf": "https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.800-82r3.pdf",
    "nist_csf_2_0.pdf":    "https://nvlpubs.nist.gov/nistpubs/CSWP/NIST.CSWP.29.pdf",
}

# Download each PDF once (skip if already present).
for fname, url in PDFS.items():
    dest = CORPUS_DIR / fname
    if dest.exists():
        print(f"✅ already downloaded: {fname}")
        continue
    resp = requests.get(url, headers=HEADERS, timeout=120)
    resp.raise_for_status()
    dest.write_bytes(resp.content)
    print(f"✅ saved {fname} ({len(resp.content)/1e6:.1f} MB)")

In [ ]:
from bs4 import BeautifulSoup
import json, time

# Fetch a sample of CISA ICS advisories from their RSS feed.
N_ADVISORIES = 20
FEED_URL = "https://www.cisa.gov/cybersecurity-advisories/ics-advisories.xml"

advisories = []
try:
    feed = requests.get(FEED_URL, headers=HEADERS, timeout=60)
    feed.raise_for_status()
    items = BeautifulSoup(feed.content, "xml").find_all("item")[:N_ADVISORIES]
    for it in items:
        title, link = it.title.get_text(strip=True), it.link.get_text(strip=True)
        try:
            page = requests.get(link, headers=HEADERS, timeout=60)
            page.raise_for_status()
            soup = BeautifulSoup(page.content, "lxml")
            main = soup.find("main") or soup.body
            text = " ".join(main.get_text(" ", strip=True).split())
            if len(text) > 500:
                advisories.append({"title": title, "url": link, "text": text})
            time.sleep(1)  # be polite to CISA
        except Exception as e:
            print(f"  ⚠️ skipped {link}: {e}")
except Exception as e:
    print("⚠️ Could not fetch the CISA feed:", e)

# Fallback advisories so the pipeline always runs end-to-end.
if len(advisories) < 3:
    advisories = [
        {"title": "ICSA-FALLBACK-01: Example PLC Hardcoded Credentials", "url": "fallback://01",
         "text": "Example PLC family, CVSS 9.8. Hardcoded credentials (CWE-798) let a remote attacker modify control logic. Mitigations: update firmware, isolate control networks behind firewalls, use VPNs for remote access."},
        {"title": "ICSA-FALLBACK-02: Example HMI Path Traversal", "url": "fallback://02",
         "text": "Example HMI product, path traversal (CWE-22), CVSS 7.5, allows reading arbitrary files. Mitigations: upgrade, restrict network access, monitor file access, apply defense-in-depth."},
        {"title": "ICSA-FALLBACK-03: Example Historian SQL Injection", "url": "fallback://03",
         "text": "Example historian server, SQL injection (CWE-89), CVSS 8.6, in the web reporting interface. Mitigations: apply hotfix, enforce least privilege, audit logs, segment historians in a DMZ."},
    ]

# Save the advisories alongside the PDFs.
(CORPUS_DIR / "cisa_advisories.json").write_text(json.dumps(advisories, indent=2))
print(f"✅ corpus ready: 2 NIST PDFs + {len(advisories)} CISA advisories")

## Step 2 — Parse & chunk the documents

Extract text per PDF page, then cut everything into fixed-size character chunks with overlap. Each chunk keeps `source` + `page` metadata so answers can cite their origin.

In [ ]:
from pypdf import PdfReader
import json

def extract_pdf_pages(path):
    # Return one {source, page, text} dict per non-empty page.
    reader = PdfReader(str(path))
    pages = []
    for i, pg in enumerate(reader.pages, start=1):
        text = (pg.extract_text() or "").strip()
        if len(text) > 80:  # skip near-empty pages
            pages.append({"source": path.name, "page": i, "text": " ".join(text.split())})
    return pages

# Collect all document units (PDF pages + advisories).
documents = []
for fname in PDFS:
    pages = extract_pdf_pages(CORPUS_DIR / fname)
    documents.extend(pages)
    print(f"✅ {fname}: {len(pages)} pages of text")

for adv in json.loads((CORPUS_DIR / "cisa_advisories.json").read_text()):
    documents.append({"source": f"CISA: {adv['title']}", "page": 1, "text": adv["text"]})
print(f"✅ total document units: {len(documents)}")

In [ ]:
CHUNK_SIZE = 1000     # characters per chunk
CHUNK_OVERLAP = 150   # characters shared between neighbouring chunks

def naive_chunk(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    # Simple fixed-size character chunking with overlap.
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks

# Build parallel lists of chunk text and its metadata.
chunks, metadatas = [], []
for doc in documents:
    for piece in naive_chunk(doc["text"]):
        chunks.append(piece)
        metadatas.append({"source": doc["source"], "page": doc["page"]})

print(f"✅ {len(chunks)} chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")

## Step 3 — Embed & index in ChromaDB

Embed every chunk with `all-MiniLM-L6-v2` and store the vectors + metadata in a persistent ChromaDB collection. This is the slow cell; it only needs to run once per runtime.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

PERSIST_DIR = "./chroma_db"
EMBED_MODEL = "all-MiniLM-L6-v2"

# Load the embedding model and open a persistent Chroma collection.
embedder = SentenceTransformer(EMBED_MODEL)
chroma = chromadb.PersistentClient(path=PERSIST_DIR)
collection = chroma.get_or_create_collection("secureops", metadata={"hnsw:space": "cosine"})

# Embed and add chunks in batches (skip if the index is already built).
if collection.count() >= len(chunks):
    print(f"✅ index already built ({collection.count()} chunks)")
else:
    BATCH = 256
    for i in range(0, len(chunks), BATCH):
        batch_docs = chunks[i:i+BATCH]
        embs = embedder.encode(batch_docs, show_progress_bar=False).tolist()
        collection.add(
            ids=[f"chunk-{j}" for j in range(i, i + len(batch_docs))],
            documents=batch_docs,
            embeddings=embs,
            metadatas=metadatas[i:i+BATCH],
        )
        print(f"  indexed {min(i+BATCH, len(chunks))}/{len(chunks)}", end="\r")
    print(f"\n✅ index built: {collection.count()} chunks")

## Step 4 — Retrieve

Embed the question and return the top-*k* most similar chunks.

In [ ]:
TOP_K = 5

def retrieve(question, k=TOP_K):
    # Return the k most similar chunks as (text, metadata, distance).
    q_emb = embedder.encode([question]).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=k)
    return list(zip(res["documents"][0], res["metadatas"][0], res["distances"][0]))

# Quick look at what retrieval returns.
for text, meta, dist in retrieve("What does NIST recommend regarding remote access to OT networks?"):
    print(f"[{meta['source']} p.{meta['page']}] (distance {dist:.3f})")
    print("   ", text[:160].replace("\n", " "), "...\n")

## Step 5 — Generate a grounded, cited answer

Pass the retrieved chunks to Gemini as numbered context blocks. The prompt enforces: cite every claim, and refuse when the context doesn't contain the answer.

In [ ]:
import time

# System prompt: grounding + citation + refusal rules.
SYSTEM_PROMPT = """You are SecureOps Assistant, helping a junior security analyst understand industrial (OT/ICS) cybersecurity.

Rules:
1. Answer ONLY using the numbered context blocks. Do not use outside knowledge.
2. Cite the supporting block number(s) after each claim, like [1] or [2][3].
3. If the context does not contain the answer, reply exactly:
   "I don't have enough information in my knowledge base to answer that."
4. Be concise. Never invent products, numbers, or recommendations."""

def build_prompt(question, hits):
    # Format retrieved chunks as numbered, source-labelled context blocks.
    blocks = "\n\n".join(
        f"[{i}] (source: {meta['source']}, page {meta['page']})\n{text}"
        for i, (text, meta, _) in enumerate(hits, start=1)
    )
    return f"{SYSTEM_PROMPT}\n\n=== CONTEXT BLOCKS ===\n{blocks}\n\n=== QUESTION ===\n{question}\n\n=== ANSWER ==="

def ask_secureops(question, k=TOP_K, retries=3):
    # Retrieve, build the prompt, then generate (with a simple retry for rate limits).
    hits = retrieve(question, k)
    prompt = build_prompt(question, hits)
    for attempt in range(retries):
        try:
            resp = client.models.generate_content(model=GEN_MODEL, contents=prompt)
            answer = resp.text.strip()
            break
        except Exception as e:
            wait = 20 * (attempt + 1)
            print(f"  ⚠️ API error ({e}); retrying in {wait}s ...")
            time.sleep(wait)
    else:
        return "ERROR: generation failed after retries."

    # Append the list of sources that were retrieved.
    sources = "\n".join(
        f"  [{i}] {meta['source']} (p.{meta['page']})"
        for i, (_, meta, _) in enumerate(hits, start=1)
    )
    return f"{answer}\n\n--- Sources retrieved ---\n{sources}"

print(ask_secureops("What does NIST recommend regarding remote access to OT networks?"))

## Step 6 — Evaluation

A small test set with three metrics:
- **Retrieval**: hit@k + MRR (does an answer-bearing chunk get retrieved?)
- **Groundedness**: LLM-as-judge checks every claim is supported by the retrieved context
- **Honesty**: unanswerable questions must trigger the refusal

The per-question table at the end is what you screenshot for the pitch.

In [ ]:
import json, time, re
import pandas as pd

# === CELL A — the test set ===
# ~70% answerable / ~30% unanswerable, mixing NIST 800-82, CSF, and CISA advisories.
#
# gold_source : substring expected in a retrieved chunk's source metadata
# gold_phrase : distinctive string expected in the answer-bearing chunk
#               (str, or a list = any-of). Content-based so it survives re-chunking.

EVAL_SET = [
    # --- NIST SP 800-82r3 ---
    {"id": "n1", "type": "answerable",
     "question": "What does NIST recommend regarding remote access to OT networks?",
     "gold_source": "nist_sp800_82", "gold_phrase": ["remote access", "vpn"]},
    {"id": "n2", "type": "answerable",
     "question": "How do OT security priorities differ from IT security priorities?",
     "gold_source": "nist_sp800_82", "gold_phrase": ["availability", "safety"]},
    {"id": "n3", "type": "answerable",
     "question": "How should an OT network be segmented from the corporate IT network?",
     "gold_source": "nist_sp800_82", "gold_phrase": ["segment", "dmz", "zone"]},
    {"id": "n4", "type": "answerable",
     "question": "What is the defense-in-depth strategy for OT security?",
     "gold_source": "nist_sp800_82", "gold_phrase": ["defense-in-depth", "layer"]},
    {"id": "n5", "type": "answerable",
     "question": "What firewall policy does NIST recommend between IT and OT networks?",
     "gold_source": "nist_sp800_82", "gold_phrase": ["firewall", "deny"]},
    {"id": "n6", "type": "answerable",
     "question": "Why is least privilege important for OT access control?",
     "gold_source": "nist_sp800_82", "gold_phrase": "least privilege"},
    {"id": "n7", "type": "answerable",
     "question": "How should wireless communications in OT environments be protected?",
     "gold_source": "nist_sp800_82", "gold_phrase": ["wireless", "encrypt"]},
    {"id": "n8", "type": "answerable",
     "question": "What physical security measures does NIST recommend for OT assets?",
     "gold_source": "nist_sp800_82", "gold_phrase": ["physical"]},
    {"id": "n9", "type": "answerable",
     "question": "How should patch and vulnerability management be handled in OT systems?",
     "gold_source": "nist_sp800_82", "gold_phrase": ["patch", "vulnerab"]},

    # --- NIST CSF 2.0 ---
    {"id": "f1", "type": "answerable",
     "question": "What are the core functions of the NIST Cybersecurity Framework?",
     "gold_source": "nist_csf",
     "gold_phrase": ["govern", "identify", "protect", "detect", "respond", "recover"]},
    {"id": "f2", "type": "answerable",
     "question": "What is the purpose of the GOVERN function in the NIST CSF?",
     "gold_source": "nist_csf", "gold_phrase": ["govern"]},

    # --- CISA ICS advisories ---
    # NOTE: the CISA feed is live, so the exact advisories change each run.
    # Rockwell Automation & Schneider Electric appear in nearly every ICS feed pull.
    # For a tighter test, open corpus/cisa_advisories.json, pick one advisory, and
    # replace gold_phrase with a distinctive token from it (a product name or CVE).
    {"id": "c1", "type": "answerable",
     "question": "What CISA advisory affects Rockwell Automation products and what is recommended?",
     "gold_source": "CISA", "gold_phrase": "rockwell"},
    {"id": "c2", "type": "answerable",
     "question": "Summarise the CISA advisory affecting Schneider Electric products.",
     "gold_source": "CISA", "gold_phrase": "schneider"},

    # --- Honesty set: answer is NOT in the corpus -> must refuse ---
    {"id": "h1", "type": "unanswerable",
     "question": "What is our company's firewall configuration?"},
    {"id": "h2", "type": "unanswerable",
     "question": "What is the IP address of our SCADA server?"},
    {"id": "h3", "type": "unanswerable",
     "question": "How many analysts are on our security team?"},
    {"id": "h4", "type": "unanswerable",
     "question": "What is the share price of Siemens today?"},
    {"id": "h5", "type": "unanswerable",
     "question": "When is the next scheduled maintenance window for our plant?"},
]
print(f"Eval set: {len(EVAL_SET)} items "
      f"({sum(i['type']=='answerable' for i in EVAL_SET)} answerable / "
      f"{sum(i['type']=='unanswerable' for i in EVAL_SET)} unanswerable)")

In [ ]:
# === CELL B — metric helpers ===

REFUSAL_MARK = "i don't have enough information"   # matches the SYSTEM_PROMPT refusal


def _as_list(x):
    return x if isinstance(x, list) else [x]


def retrieval_metrics(item, k=TOP_K):
    """hit@k and MRR using content-phrase matching (any-of)."""
    hits = retrieve(item["question"], k)
    texts = [t.lower() for t, _, _ in hits]
    phrases = [p.lower() for p in _as_list(item["gold_phrase"])]
    rank = 0
    for i, t in enumerate(texts, start=1):
        if any(p in t for p in phrases):
            rank = i
            break
    return (1 if rank else 0), (1.0 / rank if rank else 0.0), hits


def is_refusal(answer):
    return REFUSAL_MARK in answer.lower()


def _parse_json(raw):
    """Robustly pull a JSON object out of a model reply."""
    raw = re.sub(r"```(json)?", "", raw).strip()
    try:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        return json.loads(m.group(0)) if m else {}
    except Exception:
        return {}


def _judge(prompt, retries=3):
    for attempt in range(retries):
        try:
            return client.models.generate_content(model=GEN_MODEL, contents=prompt).text.strip()
        except Exception as e:
            time.sleep(15 * (attempt + 1))
    return ""


def judge_grounded(question, answer, context):
    """LLM-as-judge faithfulness. Returns 1 if every claim is supported by context."""
    prompt = f"""You are a strict evaluator. Decide if the ANSWER is fully supported by the CONTEXT.
Output ONLY JSON: {{"grounded": 1 or 0, "reason": "<one short sentence>"}}
Rules:
- grounded=1 only if every factual claim in the ANSWER appears in, or follows directly from, the CONTEXT.
- grounded=0 if any claim is unsupported, drawn from outside knowledge, or contradicts the CONTEXT.
- A correct refusal ("I don't have enough information...") counts as grounded=1.

CONTEXT:
{context}

QUESTION: {question}
ANSWER: {answer}
"""
    res = _parse_json(_judge(prompt))
    return int(res.get("grounded", 0)), res.get("reason", "")

In [ ]:
# === CELL C — run it ===
# pause respects the free-tier rate limit. Each answerable item = 1 generation +
# 1 judge call; unanswerable = 1 generation. Keep a few seconds between calls.

def run_eval(eval_set=EVAL_SET, k=TOP_K, judge=True, pause=7):
    rows = []
    for item in eval_set:
        ans_full = ask_secureops(item["question"], k=k)
        answer = ans_full.split("--- Sources retrieved ---")[0].strip()

        if item["type"] == "unanswerable":
            rows.append({
                "id": item["id"], "type": "unanswerable",
                "hit@k": None, "mrr": None,
                "grounded": int(is_refusal(answer)),   # correct refusal == grounded
                "honest": int(is_refusal(answer)),
                "note": "refused" if is_refusal(answer) else "FAILED TO REFUSE",
            })
        else:
            hit, mrr, hits = retrieval_metrics(item, k)
            grounded, reason = (judge_grounded(
                item["question"], answer,
                "\n\n".join(t for t, _, _ in hits)) if judge else (None, ""))
            rows.append({
                "id": item["id"], "type": "answerable",
                "hit@k": hit, "mrr": round(mrr, 3),
                "grounded": grounded, "honest": None, "note": reason[:60],
            })
        time.sleep(pause)
    return pd.DataFrame(rows)


def summarise(df):
    ans = df[df.type == "answerable"]
    un = df[df.type == "unanswerable"]
    print("=== SecureOps evaluation ===")
    print(f"answerable items   : {len(ans)}")
    print(f"  hit@{TOP_K}        : {ans['hit@k'].mean():.2f}")
    print(f"  MRR             : {ans['mrr'].mean():.2f}")
    if ans['grounded'].notna().any():
        print(f"  groundedness    : {ans['grounded'].mean():.2f}")
    print(f"unanswerable items : {len(un)}")
    if len(un):
        print(f"  refusal accuracy: {un['honest'].mean():.2f}")
    return df

In [ ]:
# Run the evaluation (takes a couple of minutes due to rate-limit pauses).
results = run_eval()
summarise(results)
results            # full per-question table — screenshot this for the pitch